# Beam Geometry And Fields

This notebook reconstructs the 780 nm cooling-beam geometry and calculates the spatial intensity structure using the current apparatus defaults. The goal is to verify the cooling-beam layout independently before adding any 1529 nm trapping-beam physics.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
SRC_PATH = PROJECT_ROOT / 'src'
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

from pmot import (
    build_cooling_beams,
    default_simulation_config,
    plot_apparatus_geometry_3d,
    plot_beam_crossing_zoom,
    plot_intensity_cloud_3d,
    plot_intensity_lineout,
    plot_scalar_field_slice,
    project_paths,
    sample_intensity_along_line,
    sample_intensity_cloud,
    sample_intensity_slice,
)


In [ ]:
PATHS = project_paths(PROJECT_ROOT)
CONFIG = default_simulation_config()
COOLING_BEAMS = build_cooling_beams(CONFIG)

pd.DataFrame([
    {
        'label': beam.label,
        'axis': beam.axis_name,
        'wavelength_nm': 1e9 * beam.wavelength_m,
        'waist_um': 1e6 * beam.waist_radius_m,
        'power_w': beam.power_w,
        'waist_position_mm': tuple(1e3 * value for value in beam.waist_position_m),
    }
    for beam in COOLING_BEAMS
]).head()

In [ ]:
x_mm, x_intensity = sample_intensity_along_line(COOLING_BEAMS, (-25e-3, 0.0, 0.0), (25e-3, 0.0, 0.0))
y_mm, y_intensity = sample_intensity_along_line(COOLING_BEAMS, (0.0, -25e-3, 0.0), (0.0, 25e-3, 0.0))
z_mm, z_intensity = sample_intensity_along_line(COOLING_BEAMS, (0.0, 0.0, -25e-3), (0.0, 0.0, 25e-3))

plot_intensity_lineout(x_mm, x_intensity, 'Cooling Intensity Along x', path=PATHS['outputs_fields'] / 'cooling_lineout_x.png');
plot_intensity_lineout(y_mm, y_intensity, 'Cooling Intensity Along y', path=PATHS['outputs_fields'] / 'cooling_lineout_y.png');
plot_intensity_lineout(z_mm, z_intensity, 'Cooling Intensity Along z', path=PATHS['outputs_fields'] / 'cooling_lineout_z.png');

In [ ]:
plot_beam_crossing_zoom(x_mm, x_intensity, path=PATHS['outputs_fields'] / 'cooling_beam_crossing_zoom.png');

In [ ]:
for plane in ('xy', 'xz', 'yz'):
    axes_1, axes_2, grid = sample_intensity_slice(
        COOLING_BEAMS,
        plane=plane,
        extent_m=CONFIG.volume_extent_m,
        samples_per_axis=CONFIG.samples_per_axis,
    )
    plot_scalar_field_slice(
        axes_1,
        axes_2,
        grid,
        plane=plane,
        title=f'Cooling Intensity Slice: {plane.upper()}',
        colorbar_label='Intensity [W/m$^2$]',
        path=PATHS['outputs_fields'] / f'cooling_intensity_slice_{plane}.png',
        log_scale=True,
    );

In [ ]:
CLOUD = sample_intensity_cloud(COOLING_BEAMS, axial_extent_m=30e-3, axial_samples=17, radial_rings=3, angular_samples=12)
plot_intensity_cloud_3d(CLOUD, title='3D Cooling-Field Intensity Cloud', path=PATHS['outputs_figures'] / 'cooling_intensity_cloud_3d.png');
plot_apparatus_geometry_3d(COOLING_BEAMS, path=PATHS['outputs_figures'] / 'cooling_apparatus_geometry_3d.png');